# Lecture 09: REINFORCE and PPO

E01–E04 consolidate the taught material. P01–P02 are GRPO/DPO supplements, not Quiz 4 prerequisites.

Ungraded practice. Python 3.11+, standard library only; offline CPU, 0 GPU-hours. Predict each result before running the checked example. All numerical data are synthetic.


In [ ]:
import math
from statistics import mean, pstdev


## E01 · 6 minutes · REINFORCE mean

For p=sigmoid(theta)=1/4, rewards 1 and 0, verify the expected score-function gradient with baselines 0 and 1/2. Does the mean change?

The following cell checks the worked answer.


In [ ]:
p = 0.25
for baseline in (0, 0.5):
    expected = p*(1-baseline)*(1-p) + (1-p)*(0-baseline)*(-p)
    assert math.isclose(expected, 3/16)
    print(baseline, expected)


## E02 · 7 minutes · PPO signs

Use epsilon=.2. For ratios .7 and 1.3 and advantages +2 and -2, compare the unclipped objective with the clipped minimum. This objective is maximized.

The following cell checks the worked answer.


In [ ]:
def clipped_objective(ratio, advantage, epsilon=0.2):
    clipped = min(max(ratio, 1-epsilon), 1+epsilon)
    return min(ratio*advantage, clipped*advantage)
expected = {(0.7, 2):1.4, (1.3, 2):2.4, (0.7, -2):-1.6, (1.3, -2):-2.6}
for case, answer in expected.items():
    value = clipped_objective(*case)
    assert math.isclose(value, answer)
    print(case, value)


## E03 · 7 minutes · GAE backward recursion

Let rewards be [0,0,1], values [.2,.3,.4], terminal value 0, gamma=1 and lambda=.9. Compute deltas, advantages and value targets.

The following cell checks the worked answer.


In [ ]:
rewards, values = [0, 0, 1], [0.2, 0.3, 0.4]
deltas = [r + nxt - value for r, nxt, value in zip(rewards, values[1:]+[0], values)]
advantages = [0.0] * len(deltas)
carry = 0.0
for t in reversed(range(len(deltas))):
    carry = deltas[t] + 0.9*carry
    advantages[t] = carry
targets = [a+v for a,v in zip(advantages,values)]
assert all(math.isclose(a,b) for a,b in zip(advantages,[.676,.64,.6]))
assert all(math.isclose(a,b) for a,b in zip(targets,[.876,.94,1]))
print({'deltas':deltas, 'advantages':advantages, 'targets':targets})


## E04 · 5 minutes · Preserve the denominator

A rollout recorded probability .20; the reference is .10; current probability becomes .24. An implementation overwrites old with .24. Calculate the correct and erroneous PPO ratios.

The following cell checks the worked answer.


In [ ]:
old, reference, current = .20, .10, .24
assert math.isclose(current/old, 1.2)
assert current/current == 1
print({'correct_ratio':current/old, 'overwritten_ratio':current/current,
       'reference_log_ratio':math.log(current/reference)})
print('Cache rollout log probabilities, not a live model output. Refresh only with a new rollout batch.')


## P01 · optional · GRPO group statistics

For group rewards [0,0,1,1], use population standard deviation. What happens if all rewards are 1? Does relative normalization require a learned critic?

The following cell checks the worked answer.


In [ ]:
def group_advantages(rewards):
    center, scale = mean(rewards), pstdev(rewards)
    return [0.0 for _ in rewards] if scale == 0 else [(r-center)/scale for r in rewards]
assert group_advantages([0,0,1,1]) == [-1,-1,1,1]
assert group_advantages([1,1,1,1]) == [0,0,0,0]
print(group_advantages([0,0,1,1]))
print('No learned critic is required; a reward source still is. Zero variance supplies no relative preference signal.')


## P02 · optional · DPO margin

Let the policy chosen/rejected sequence log probabilities be -2/-3, reference -2.5/-3, and beta=.2. Compute the DPO logit, loss and derivative with respect to the policy log-probability gap.

The following cell checks the worked answer.


In [ ]:
beta = 0.2
policy_gap, reference_gap = -2-(-3), -2.5-(-3)
z = beta*(policy_gap-reference_gap)
loss = math.log1p(math.exp(-z))
derivative = -beta/(1+math.exp(z))
assert math.isclose(z, .1)
def objective(gap):
    return math.log1p(math.exp(-beta*(gap-reference_gap)))
h = 1e-6
finite_difference = (objective(policy_gap+h)-objective(policy_gap-h))/(2*h)
assert math.isclose(finite_difference, derivative, abs_tol=1e-9)
print({'logit':z,'loss':loss,'derivative':derivative})
